# 🧬 Benchmark Comparatif : WhatsHap, HapCUT2 et Algorithme Spectral/MCMC sur Hi-C NA12878 (chr22)

Ce notebook réalise une comparaison systématique et rigoureuse entre quatre approches de phasing d'haplotypes sur des données réelles de contacts Hi-C (GM12878) pour le chromosome 22 :
1. **Baseline Spectral :** Notre clustering spectral signé global (relaxation continue du problème de coupe signée sur le graphe de reads).
2. **Gibbs MCMC :** Notre échantillonnage de Gibbs k-hop local pour raffiner la cohérence physique.
3. **WhatsHap (SOTA exact) :** L'état de l'art résolvant le wMEC (Weighted Minimum Error Correction) de manière exacte par programmation dynamique.
4. **HapCUT2 (SOTA par coupe de graphe) :** L'algorithme de coupe de graphe de variants itérative, de référence pour les liaisons à longue portée.

Toutes nos fonctions de calcul et de parsing d'allèles sont importées depuis le module optimisé `src/` (accéléré avec Numba JIT et SciPy sparse).

In [1]:
# @title 🛠️ Configuration de l'environnement et imports
import sys
import os
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt

# Ajouter la racine du dépôt pour importer src
sys.path.append(os.path.abspath(".."))
import src

print("✅ Environnement initialisé avec succès !")

In [2]:
# @title 🎛️ Formulaire de Configuration des Paramètres Globaux
chromosome = "chr22" #@param ["chr22"] {type:"string"}
beta_safety = 0.5 #@param {type:"number"}
mcmc_steps = 10000000 #@param {type:"integer"}
beta = 1.0 #@param {type:"number"}
verbose = True #@param {type:"boolean"}

print("✨ Paramètres configurés avec succès !")
print(f"  - Chromosome cible : {chromosome}")
print(f"  - Nombre de pas MCMC : {mcmc_steps}")
print(f"  - Température inverse (beta) : {beta}")

In [3]:
# @title 🧬 Téléchargement et extraction des données réelles Hi-C (GM12878)
vcf_phased = f"HG001_phased_{chromosome}.vcf"
vcf_unphased = f"HG001_unphased_{chromosome}.vcf"
bam_path = f"NA12878_{chromosome}_hic.bam"

# 1. VCF Phased & Unphased de GIAB
VCF_URL = "https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/giab/release/NA12878_HG001/NISTv4.2.1/GRCh38/SupplementaryFiles/HG001_GRCh38_1_22_v4.2.1_benchmark_hifiasm_v11_phasetransfer.vcf.gz"
TBI_URL = VCF_URL + ".tbi"
BED_URL = "https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/giab/release/NA12878_HG001/NISTv4.2.1/GRCh38/HG001_GRCh38_1_22_v4.2.1_benchmark.bed"

print("🧬 Préparation du VCF de vérité terrain et de confiance...")
if not os.path.exists("HG001_phased.vcf.gz"):
    src.download_file(VCF_URL, "HG001_phased.vcf.gz")
    src.download_file(TBI_URL, "HG001_phased.vcf.gz.tbi")
if not os.path.exists("HG001_benchmark.bed"):
    src.download_file(BED_URL, "HG001_benchmark.bed")

src.prepare_cleaned_vcfs("HG001_phased.vcf.gz", vcf_unphased, vcf_phased, chromosome)
variants = src.load_variants_dict(vcf_phased, chromosome)
print(f"  -> {len(variants)} variants hétérozygotes de vérité terrain chargés.")

# 2. Contacts Hi-C ENCODE GM12878 Profonds (5.9 Go)
if not os.path.exists(bam_path):
    PAIRS_URL = "https://www.encodeproject.org/files/ENCFF527IWE/@@download/ENCFF527IWE.pairs.gz"
    pairs_local = "ENCFF527IWE.pairs.gz"
    pairs_filtered = f"ENCFF527IWE_{chromosome}.pairs.gz"
    
    src.download_file(PAIRS_URL, pairs_local)
    if not os.path.exists(pairs_filtered):
        print(f"⏳ Extraction ultra-rapide des contacts {chromosome} (7 millions de paires)...")
        os.system(f"zcat {pairs_local} | grep -E '^#|{chromosome}' | gzip > {pairs_filtered}")
        if os.path.exists(pairs_local):
            os.remove(pairs_local)
            print("🧹 Fichier volumineux de 5.9 Go supprimé pour libérer l'espace.")
            
    # Générer le BAM ultra-rapide (reads de 1 pb pour désactiver le goulot PairHMM)
    # max_pairs = 10 000 000 000 000 pour n'avoir aucune limite artificielle de connectivité
    src.build_hic_bam_from_pairs(pairs_filtered, bam_path, variants, max_pairs=10000000000000)

print("✅ Toutes les données sont prêtes !")

In [4]:
# @title ⏳ Extraction ultra-rapide des profils de reads (pysam)
t0 = time.time()
read_profiles = src.extract_read_profiles_fast(bam_path, variants, chromosome)
print(f"  -> {len(read_profiles)} profils de reads extraits en {time.time() - t0:.2f}s.")

In [5]:
# @title ⏳ Construction de la vérité terrain et du Graphe signé d'interactions
# Détermination des spins de vérité terrain des reads
true_spins = {}
for rid, profile in read_profiles.items():
    votes = []
    for pos, (allele_val, _) in profile.items():
        if pos in variants:
            true_gt = variants[pos]["gt"]
            if allele_val == true_gt[0]:
                votes.append(1)
            elif allele_val == true_gt[1]:
                votes.append(-1)
    if len(votes) > 0:
        true_spins[rid] = 1 if np.sum(votes) >= 0 else -1

active_reads = {rid: prof for rid, prof in read_profiles.items() if rid in true_spins}
read_ids = sorted(active_reads.keys(), key=lambda rid: min(active_reads[rid].keys()))
R = len(read_ids)
read_id_to_idx = {rid: i for i, rid in enumerate(read_ids)}
true_spin_vec = np.array([true_spins[rid] for rid in read_ids])

print(f"  -> {R} reads actifs restants pour le phasing.")

# Construction du graphe avec Numba et SciPy CSR
t0 = time.time()
# max_reads_per_variant=150 pour éliminer les OOM sur les hotspots répétitifs
W = src.build_signed_graph(active_reads, variants, read_id_to_idx, beta_safety=beta_safety, max_reads_per_variant=150)
print(f"✅ Graphe construit (CSR sparse) de taille {W.shape[0]}x{W.shape[1]} en {time.time() - t0:.2f}s.")

In [6]:
# @title 🚀 Clustering Spectral Signé Baseline
t0 = time.time()
# use_gpu=True pour utiliser CuPy si disponible
v_eigen = src.signed_spectral_clustering(W, type="unnormalized", use_gpu=True)
pred_baseline = np.sign(v_eigen)
print(f"✅ Clustering Spectral Signé terminé en {time.time() - t0:.2f}s.")

corr = np.mean(pred_baseline == true_spin_vec)
if corr < 0.5:
    pred_baseline = -pred_baseline
    corr = 1 - corr
print(f"  -> Précision du clustering spectral baseline : {corr:.2%}")

In [7]:
# @title ⏳ Lancement du Gibbs Sampling MCMC k-hop
t0 = time.time()
structures = src.build_structures_fast(R, W)
incident_offsets, incident_left, incident_right, incident_weight = (
    structures[4], structures[5], structures[6], structures[7]
)

print("⏳ Lancement de la MCMC k-hop Gibbs sampling (JIT Numba)...")
post_probs = src.mcmc_gibbs_sampling_numba(
    R, incident_offsets, incident_left, incident_right, incident_weight,
    pred_baseline.astype(np.float64), mcmc_steps, beta
)
pred_mcmc = np.where(post_probs >= 0.5, 1.0, -1.0)
print(f"✅ MCMC terminée en {time.time() - t0:.2f}s.")

corr_mcmc = np.mean(pred_mcmc == true_spin_vec)
if corr_mcmc < 0.5:
    pred_mcmc = -pred_mcmc
    corr_mcmc = 1 - corr_mcmc
print(f"  -> Précision de la MCMC k-hop : {corr_mcmc:.2%}")

In [8]:
# @title ⏳ Exportation des fichiers VCF phasés pour la Baseline et la MCMC
variant_ps = {}
for pos in variants.keys():
    variant_ps[pos] = 1  # Un seul bloc global

print("⏳ Écriture de phased_baseline.vcf...")
src.write_phased_vcf(vcf_unphased, "phased_baseline.vcf", active_reads, read_id_to_idx, pred_baseline, variants, variant_ps, chromosome)

print("⏳ Écriture de phased_mcmc.vcf...")
src.write_phased_vcf(vcf_unphased, "phased_mcmc.vcf", active_reads, read_id_to_idx, pred_mcmc, variants, variant_ps, chromosome)

print("✅ VCFs exportés avec succès !")

## 🏁 Exécution et Intégration des États de l'Art (WhatsHap et HapCUT2)

Nous allons à présent exécuter les deux outils de référence du domaine sur le même fichier BAM d'interactions réelles du chromosome 22.

In [9]:
# @title ⏳ Exécution de WhatsHap (SOTA exact)
with pysam.VariantFile(vcf_unphased) as vcf:
    sample_name = list(vcf.header.samples)[0]
print(f"⏳ Exécution de WhatsHap pour le phasing de référence sur {chromosome}...")
# --ignore-read-groups et --only-snvs pour assurer la comparabilité
!whatshap phase --chromosome {chromosome} --ignore-read-groups --only-snvs --no-reference -o phased_whatshap.vcf {vcf_unphased} {bam_path}
print("✅ WhatsHap terminé avec succès !")

In [10]:
# @title ⏳ Exécution de HapCUT2 (SOTA par coupe de graphe)
print("⏳ Étape 1 : Extraction des fragments Hi-C du BAM avec extractHAIRS...")
# --hic 1 indique à extracthairs de coupler les variants de longue portée
!../HapCUT2/build/extractHAIRS --bam {bam_path} --vcf {vcf_unphased} --out hic_fragments.txt --hic 1

print("\n⏳ Étape 2 : Phasing du graphe avec HAPCUT2...")
!../HapCUT2/build/HAPCUT2 --fragments hic_fragments.txt --vcf {vcf_unphased} --output phased_hapcut2.vcf
print("✅ HapCUT2 terminé avec succès !")

## 📊 Évaluation et Synthèse Comparative Globale

Nous appelons la commande `whatshap compare` pour évaluer de manière équitable les quatre approches par rapport au VCF phased de vérité terrain de GIAB.

In [11]:
# @title ⏳ Lancement de whatshap compare et parsing des logs
print("⏳ Comparaison de phased_baseline.vcf...")
!whatshap compare --sample {sample_name} {vcf_phased} phased_baseline.vcf > compare_baseline.txt
metrics_base = src.parse_whatshap_compare("compare_baseline.txt", "phased_baseline.vcf", chromosome)

print("⏳ Comparaison de phased_mcmc.vcf...")
!whatshap compare --sample {sample_name} {vcf_phased} phased_mcmc.vcf > compare_mcmc.txt
metrics_mcmc = src.parse_whatshap_compare("compare_mcmc.txt", "phased_mcmc.vcf", chromosome)

print("⏳ Comparaison de phased_whatshap.vcf...")
!whatshap compare --sample {sample_name} {vcf_phased} phased_whatshap.vcf > compare_whatshap.txt
metrics_whatshap = src.parse_whatshap_compare("compare_whatshap.txt", "phased_whatshap.vcf", chromosome)

print("⏳ Comparaison de phased_hapcut2.vcf...")
!whatshap compare --sample {sample_name} {vcf_phased} phased_hapcut2.vcf > compare_hapcut2.txt
metrics_hapcut2 = src.parse_whatshap_compare("compare_hapcut2.txt", "phased_hapcut2.vcf", chromosome)

# Construire la table de synthèse
results = []
results.append({"Méthode": "Baseline Spectral", **metrics_base})
results.append({"Méthode": "Gibbs MCMC k-hop", **metrics_mcmc})
results.append({"Méthode": "WhatsHap (SOTA exact)", **metrics_whatshap})
results.append({"Méthode": "HapCUT2 (SOTA graphe)", **metrics_hapcut2})

df_res = pd.DataFrame(results)

print("\n=========================================================================")
print("📊 TABLEAU COMPARATIF DES RÉSULTATS SUR NA12878 CHR22 (Hi-C RÉEL)")
print("=========================================================================")
display(df_res)

# Supprimer les fichiers de logs temporaires
for f_txt in ["compare_baseline.txt", "compare_mcmc.txt", "compare_whatshap.txt", "compare_hapcut2.txt", "hic_fragments.txt"]:
    if os.path.exists(f_txt):
        os.remove(f_txt)